In [1]:
import os
import pandas as pd
import numpy as np
import time
import random
from understatapi import UnderstatClient

In [2]:
# フォルダーの自動生成
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

In [3]:
BIG_6 = ["Arsenal", "Chelsea", "Liverpool", "Manchester City", "Manchester United", "Tottenham"]
SEASONS= ["2021", "2022", "2023", "2024", "2025"]

sleep_time = random.uniform(1.0, 3.0)

## 1. チーム別・時間帯（Timing）データの取得と保存

In [6]:
big_six_stats = []

with UnderstatClient() as understat:
    for season in SEASONS:        
        for team in BIG_6:
            try:
                context_data = understat.team(team=team).get_context_data(season=season)
                timing_dict = context_data.get("timing", {})
        
                for interval, stats in timing_dict.items():
                    row = {
                        "season": season,
                        "team": team,
                        "interval": interval,
                        "shots": stats.get("shots", 0),
                        "goals": stats.get("goals", 0),
                        "xG": stats.get("xG", 0),
                        "shots_against": stats.get("against", {}).get("shots", 0),
                        "goals_against": stats.get("against", {}).get("goals", 0),
                        "xGA": stats.get("against", {}).get("xG", 0)
                    }
        
                    big_six_stats.append(row)
                    time.sleep(sleep_time)

            except Exception as e:
                print(f"Errors fetching {season} {team} :{e}")

df_raw = pd.DataFrame(big_six_stats)
df_raw.to_csv("data/raw/raw_interval_data.csv", index=False)

## 2. チーム別・試合状況（Game State）データの取得と保存
**目的：試合中のスコア状況（勝っているか、引き分けているか、負けいているか）」を中心とした試合の文脈を探る。状況によってチームの戦術や選手の心理、プレー選択が大きく変化する（スコア効果）。**
- 負けている状態：リスクを冒しても前がかりになり、シュート数や攻撃エリアへの侵入が増える傾向にある。
- 勝っている状態：無理に責める必要がないため、ブロックを敷いて守備に徹したり、ポゼッション（ボール保持）で時間を消化する。
- 引き分けの状態：試合開始時や均衡した状態であり、両チームが本来用意してきた「ゲームモデル（理想の戦術）」が色濃く出る。

In [4]:
game_states = []

with UnderstatClient() as understat:   
    for season in SEASONS:
        for team in BIG_6:
            try:
                context_data = understat.team(team=team).get_context_data(season=season)
                game_state_dict = context_data.get("gameState", {})

                for game_state, stats in game_state_dict.items():
                    row = {
                        "season": season,
                        "team": team,
                        "game_state": game_state,
                        "minutes": stats.get("time", 0),
                        "shots": stats.get("shots", 0),
                        "goals": stats.get("goals", 0),
                        "xG": stats.get("xG", 0),
                        "shots_against": stats.get("against", {}).get("shots", 0),
                        "goals_against": stats.get("against", {}).get("goals", 0),
                        "xGA": stats.get("against", {}).get("xG", 0)
                    }
                    game_states.append(row)
                    time.sleep(sleep_time)

            except Exception as e:
                print(f"Error fetching {season} {team}: {e}")

df_raw_gs = pd.DataFrame(game_states)
df_raw_gs.to_csv("data/raw/raw_game_state_data.csv", index=False)